In [1]:
import anndata as ad
import pandas as pd
import numpy as np
import mofax as mfx
import duckdb as db

In [ ]:
def limpiar_drug(s):
    return (s.str.strip()
             .str.lower()
             .str.replace(' ', '_', regex=False)  
             .str.replace(r'_+', '_', regex=True)
             .str.strip('_'))

In [67]:


# --- RUTAS ---
MOFA_MODEL = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/mofa_sin_view/modelo_mofa_30factors.hdf5'
INPUT_PARQUET = "/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/datos/datos_con_placa_14/tidy_final_sin_views.parquet"
DRUG_PARQUET = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/drug.parquet'
OUTPUT_H5AD = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/mofa_sin_view/mofa_adata_sin_view.h5ad'

# 1. Cargar modelo MOFA
print("Cargando modelo MOFA...")
model = mfx.mofa_model(MOFA_MODEL)

# 2. Factores (muestras × factores)
Z = model.get_factors(df=True)
print(f"Factores: {Z.shape}")
print(f"Índice ejemplo: {Z.index[:4].tolist()}")

# 3. Metadata desde el índice de los factores
print("Parseando metadata desde índice...")
# Primero reemplazar el _ entre drug y concentración por __
Z.index = Z.index.to_series().str.replace(r'_(\d)', r'__\1', regex=True)
parts = Z.index.to_series().str.rsplit('__', n=3, expand=True)
parts.columns = ['drug', 'concentration', 'plate', 'cell_line']
parts['concentration'] = parts['concentration'].str.strip('_')

obs = pd.DataFrame({
    'drug': parts['drug'].values,
    'concentration': parts['concentration'].values,
    'plate': parts['plate'].values,
    'cell_line': parts['cell_line'].values
}, index=Z.index)

# Limpiar drug
obs['drug'] = limpiar_drug(obs['drug'])

# 4. Añadir MOA desde drug metadata
print("Añadiendo MOA...")
drug_meta = pd.read_parquet(DRUG_PARQUET)
# Limpiar drug en drug_meta igual que en obs
drug_meta['drug'] = limpiar_drug(drug_meta['drug'])
# Comprobacion todas las drogas coiciden
drogas_coinciden = obs['drug'].isin(drug_meta['drug']).all()
print(f"¿Todas las drogas en obs están en drug_meta? {drogas_coinciden}")
# Merge MOA (ajusta nombre de columna si es diferente)
if 'moa-fine' in drug_meta.columns:
    moa_map = drug_meta[['drug', 'moa-fine']].drop_duplicates('drug')
    obs = obs.merge(moa_map, on='drug', how='left')
    obs.index = Z.index  # restaurar índice tras merge
    print(f"  MOA añadido. NaN: {obs['moa-fine'].isna().sum()}")
else:
    print(f"  Columnas disponibles en drug_meta: {drug_meta.columns.tolist()}")
    print("  Ajusta el nombre de la columna MOA manualmente")

# 5. Pesos de MOFA por vista (genes × factores)
print("Extrayendo pesos MOFA...")
W = model.get_weights(df=True)    
# 6. Crear AnnData principal (muestras × factores)
print("Creando AnnData...")
adata = ad.AnnData(
    X=Z.values,
    obs=obs,
    var=pd.DataFrame(index=Z.columns)  # Factor1..Factor30
)

# 7. Guardar pesos en uns
adata.uns['mofa_weights'] = W.values
adata.uns['mofa_weights_genes'] = W.index.tolist()
adata.uns['mofa_weights_factors'] = W.columns.tolist()
adata.uns['mofa_views'] = list(model.views)

# 8. Guardar
print(f"\n{adata}")
print(f"\nobs columns: {adata.obs.columns.tolist()}")
adata.write(OUTPUT_H5AD, compression='gzip')
print(f"Guardado en {OUTPUT_H5AD}")

Cargando modelo MOFA...
Factores: (60404, 30)
Índice ejemplo: ['4EGI-1_5.0__3__A549', '9-ING-41_5.0__3__A549', 'APTO-253_5.0__3__A549', 'AT7519_5.0__3__A549']
Parseando metadata desde índice...
Añadiendo MOA...
¿Todas las drogas en obs están en drug_meta? True
  MOA añadido. NaN: 0
Extrayendo pesos MOFA...
Creando AnnData...

AnnData object with n_obs × n_vars = 60404 × 30
    obs: 'drug', 'concentration', 'plate', 'cell_line', 'moa-fine'
    uns: 'mofa_weights', 'mofa_weights_genes', 'mofa_weights_factors', 'mofa_views'

obs columns: ['drug', 'concentration', 'plate', 'cell_line', 'moa-fine']
Guardado en /mnt/lustre/scratch/nlsas/home/ulc/co/mao/mofa_sin_view/mofa_adata_sin_view.h5ad
